# 로봇 제어 실습

쓰는 명령은 `Arm`과 `load_pick_targets`입니다.

프로젝트 폴더에서 엽니다. 커널은 `AIvision`입니다. 칸은 위에서 아래로 한 번씩 실행합니다. 팔 켜기 칸을 두 번 실행하면 팔이 겹칩니다.

## 시작 전에

- 펜던트는 끄고 진행합니다. 이 노트북이 3D 화면(Meshcat)을 띄웁니다. 브라우저가 안 열리면 출력된 주소를 엽니다.
- 좌표는 로봇 기준입니다. +X 앞, +Y 왼쪽, +Z 위. 위치는 mm, 자세는 도입니다.
- `grip(0)`은 닫힘, `grip(100)`은 완전히 열림입니다.
- 이동 명령은 도착할 때까지 그 칸에서 기다립니다.
- `"rel"`은 지금 위치에 더합니다. `"abs"`는 베이스 기준 목표입니다.
- `rpy`는 더하는 각도가 아니라 목표 자세입니다.

## 1절 팔 켜기

`where()`는 지금 관절, 그리퍼, 끝 위치(`xyz_mm`), 자세(`rpy_deg`)를 돌려줍니다.

In [1]:
from motion import Arm, load_pick_targets

arm = Arm()
print(arm.where())

Arm      mode=virtual
Meshcat  http://127.0.0.1:7000/static/
{'joints_deg': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'gripper': 50.0, 'xyz_mm': [351.84, -0.5, 224.17], 'rpy_deg': [-180.0, -87.21, -90.0]}


## 2절 시작 자세와 그리퍼

`initial()`은 작업을 시작하는 자세입니다. 그리퍼는 닫혀 있습니다.

In [ ]:
arm.initial()
arm.grip(100)
print(arm.where())

## 3절 상대 이동

앞 30mm, 그다음 왼쪽 20mm와 위 15mm입니다. `speed`를 생략하면 25 mm/s입니다.

In [ ]:
arm.moveL([30, 0, 0], "rel", speed=10)
arm.moveL([0, 20, 15], "rel")
print(arm.where()["xyz_mm"])

## 4절 절대 위치와 자세

두 번째 이동은 위치와 손목 자세를 함께 지정합니다.

In [ ]:
arm.moveL([200, 0, 120], "abs")
arm.moveL([180, 40, 90], "abs", rpy=[0, 80, 0])
print(arm.where())

## 5절 지금 높이를 읽어서 올리기

In [ ]:
here = arm.where()
x, y, z = here["xyz_mm"]
arm.moveL([x, y, z + 40], "abs")
print(arm.where()["xyz_mm"])

## 6절 관절

`moveJ`는 관절을 돌립니다. 그리퍼 끝은 직선으로 가지 않습니다. `gripper=20`은 그리퍼를 지금보다 20만큼 더 엽니다.

In [ ]:
arm.moveJ([0, -40, 40, 0, -90, 0], "abs")
arm.moveJ([0, 10, 0, 0, 0, 0], "rel", gripper=20)
print(arm.where())

## 7절 홈

`home()`은 관절이 0인 L자입니다. 그리퍼는 반쯤 열립니다.

In [ ]:
arm.home()
print(arm.where())

## 8절 좌표를 적어서 집기

카메라는 쓰지 않습니다. 순서는 열기, 내리기, 닫기, 들기, 옆으로 옮기기, 내리기, 열기, 빠지기입니다.

In [ ]:
arm.initial()
arm.grip(100)
arm.moveL([0, 0, -40], "rel")
arm.grip(0)
arm.moveL([0, 0, 50], "rel")
arm.moveL([0, 80, 0], "rel")
arm.moveL([0, 0, -40], "rel")
arm.grip(100)
arm.moveL([0, 0, 40], "rel")
arm.home()

### 8-2 미션: 블록 옮기기 (7차시)

블록을 A 에서 집어 B 에 놓습니다. A · B 좌표는 펜던트 (또는 `arm.where()`) 로 그리퍼 끝을 맞춰 적은 숫자입니다. 아래 숫자는 예시라 자리마다 다릅니다.

멀리 옮길 때는 `path="joint"`, 내려갈 때는 `speed=10` 을 씁니다. 가상 팔로 먼저 하고, 강사 확인 뒤 실기로 합니다.

In [ ]:
ax, ay, az = 200, 50, 15                        # A: 블록을 집는 곳 (z 는 집는 높이)
bx, by = 180, -80                               # B: 놓는 곳

arm.initial()
arm.moveL([ax, ay, az + 40], "abs")             # A 위 40mm
arm.grip(100)                                   # 손 벌리기
arm.moveL([ax, ay, az], "abs", speed=10)        # 천천히 내려가기
arm.grip(0)                                     # 잡기
arm.moveL([ax, ay, az + 50], "abs")             # 들기
arm.moveL([bx, by, az + 50], "abs", path="joint")   # B 위로 (관절 이동)
arm.moveL([bx, by, az], "abs", speed=10)        # 천천히 내려가기
arm.grip(100)                                   # 놓기
arm.moveL([bx, by, az + 40], "abs")             # 빠지기
arm.initial()

### 8-3 오류 읽기 (7차시)

일부러 틀려 봅니다. 한 번에 **한 줄만** 맨 앞의 `#` 을 지우고 실행합니다. 오류는 빨간 글씨의 **마지막 줄** 부터 읽습니다. 확인했으면 다시 `#` 을 붙이고 다음 줄로 넘어갑니다.

꺼진 팔에 명령하면 `이미 종료된 Arm 입니다` 가 납니다. 10절에서 봅니다.

In [ ]:
arm.initial()

# arm.moveL([30, 0, 0])                         # "rel" · "abs" 를 빠뜨림
# arm.moveL("abs", [200, 0, 80])                # 좌표와 "abs" 순서가 바뀜
# arm.moveL([30, 0], "rel")                     # 숫자가 2개
# arm.moveL([30, 0, 0], "rel", speed=100)       # 속도가 범위 밖 (5~45)

## 9절 카메라로 집기

이 칸을 실행하기 전에 다른 터미널에서 `python yolo/pose/roi_cloud.py`를 켜 둡니다. 브라켓이 화면에 보여야 합니다.

`load_pick_targets(arm)`이 대기 위치와 집기 위치를 채웁니다. 내려놓는 X, Y는 `drop_x`, `drop_y`입니다.

In [ ]:
drop_x, drop_y = 150.0, -100.0
pick = load_pick_targets(arm)
z_g = pick.g_xyz[2]

arm.initial()
arm.moveL(pick.p_xyz, "abs", rpy=pick.p_rpy)
arm.grip(30)
arm.moveL(pick.g_xyz, "abs", rpy=pick.g_rpy)
arm.grip(0)
arm.moveL([0, 0, 30], "rel")
arm.moveL([drop_x, drop_y, z_g + 50], "abs", rpy=pick.place_rpy, path="joint")
arm.moveL([drop_x, drop_y, z_g], "abs", rpy=pick.place_rpy)
arm.grip(30)
arm.moveL([0, 0, 50], "rel")
arm.initial()

### 9-2 보이는 브라켓 모두 보기 (8차시)

카메라 창이 켜져 있어야 합니다. `load_all_pick_targets(arm)` 은 보이는 브라켓을 **모두**, 로봇에서 가까운 순서로 줍니다. 브라켓마다 판정 (`양품` · `불량` · `검사 안 함`), 색, 놓인 자세, 위치 (mm), 거리 (mm) 가 들어 있습니다.

순서는 `sorted` 로 직접 정합니다. 아래는 불량 먼저, 그다음 가까운 것부터입니다.

In [ ]:
from motion import load_all_pick_targets

objs = load_all_pick_targets(arm)
for o in objs:
    print(o.verdict, o.color, o.pose, o.xyz, o.distance)

# 불량 먼저, 그다음 가까운 것부터
order = sorted(objs, key=lambda o: (o.verdict != "불량", o.distance))
print([o.verdict for o in order])

### 9-3 미션: 불량 · 양품 나누기 (8차시)

화면 **왼쪽 절반** 의 브라켓을 하나씩 집어, **불량은 오른쪽 위 접시**, **양품은 오른쪽 아래 접시** 에 놓습니다. 한 개 옮길 때마다 카메라로 다시 봅니다 (집다가 다른 브라켓이 밀릴 수 있습니다).

- 접시 좌표는 펜던트 (또는 `arm.where()`) 로 그리퍼 끝을 접시 가운데에 맞춰 재서 바꿉니다. 아래 숫자는 예시입니다.
- 화면 왼쪽은 로봇의 +Y 쪽입니다. 그래서 `o.xyz[1] > PICK_Y` 인 브라켓만 집습니다. 놓은 브라켓은 화면 오른쪽에 있으니 다시 집지 않습니다.
- 접시 테두리에 걸리지 않게 집은 높이보다 30mm 위에서 놓습니다.

In [ ]:
from motion import ArmError, load_all_pick_targets

BAD_DISH = (230, -150)                          # 불량 접시 (화면 오른쪽 위)
GOOD_DISH = (110, -150)                         # 양품 접시 (화면 오른쪽 아래)
PICK_Y = 0                                      # 이보다 왼쪽 (+Y) 에 있는 브라켓만 집는다


def place(arm, pick, x, y):
    # 대기 → 집기 → 들기 → 접시 위 → 놓기 → 빠지기
    z_g = pick.g_xyz[2]
    arm.moveL(pick.p_xyz, "abs", rpy=pick.p_rpy)
    arm.grip(30)
    arm.moveL(pick.g_xyz, "abs", rpy=pick.g_rpy)
    arm.grip(0)
    arm.moveL([0, 0, 30], "rel")
    arm.moveL([x, y, z_g + 60], "abs", rpy=pick.place_rpy, path="joint")
    arm.moveL([x, y, z_g + 30], "abs", rpy=pick.place_rpy)
    arm.grip(30)
    arm.moveL([0, 0, 40], "rel")
    arm.initial()


arm.initial()
for _ in range(6):                              # 최대 6개
    try:
        objs = load_all_pick_targets(arm)
    except ArmError as exc:                     # 남은 브라켓이 없거나 카메라 창이 꺼짐
        print(exc)
        break
    objs = [o for o in objs if o.pick is not None and o.xyz[1] > PICK_Y]
    if not objs:
        print("왼쪽에 집을 브라켓이 없습니다")
        break
    o = sorted(objs, key=lambda o: (o.verdict != "불량", o.distance))[0]
    print(o.verdict, o.color, o.distance)
    if o.verdict == "불량":
        place(arm, o.pick, *BAD_DISH)
    else:
        place(arm, o.pick, *GOOD_DISH)          # 양품 · 검사 안 함 (검은 브라켓)

## 10절 끄기

다시 움직이려면 커널을 재시작하고 1절부터 실행합니다.

In [4]:
arm.close()

## 11절 실기

강사가 실기라고 한 뒤에만 합니다. 위의 가상 팔을 끈 다음, `REAL = True`로 바꾸고 이 칸만 실행합니다.

In [ ]:
REAL = False

if REAL:
    arm = Arm(mode="real")
    print(arm.where())
    arm.initial()
    arm.close()
else:
    print("실기 시간이 아니면 이 칸은 넘어갑니다.")